<p style="text-align: center; margin-bottom: 0; font-size: 24px;">EAE1106 - Métodos Computacionais para Economia</p>
<p style="text-align: center; margin-bottom: 0;">Faculdade de Economia, Administração, Contabilidade e Atuária<br>Universidade de São Paulo</p>
<p style="text-align: center; margin-top: 5px;">1º semestre de 2026<br><br>Prof. Arthur Viaro</p>

# Aula 23 — Seaborn e Plotly

**Conteúdo**

- [O que são Seaborn e Plotly?](#intro)
- [Seaborn: visualizações estatísticas](#seaborn)
  - [Temas e paletas](#temas)
  - [Gráfico de linha](#lineplot)
  - [Gráfico de dispersão](#scatterplot)
  - [Distribuições: histograma e KDE](#distribuicoes)
  - [Boxplot](#boxplot)
  - [Mapa de calor](#heatmap)
- [Plotly: gráficos interativos](#plotly)
  - [Gráfico de linha interativo](#plotly-linha)
  - [Dispersão interativa](#plotly-scatter)
  - [Barras interativas](#plotly-bar)
- [Quando usar cada biblioteca?](#comparacao)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

## O que são Seaborn e Plotly? <a id="intro"></a>

Na aula anterior, aprendemos a criar e personalizar gráficos com o **Matplotlib** — a biblioteca base de visualização em Python. Hoje veremos duas bibliotecas que ampliam esse ecossistema:

- **Seaborn** é construído diretamente sobre o Matplotlib e foi projetado para visualizações estatísticas. Com menos linhas de código, produz gráficos mais elegantes e é especialmente útil para explorar distribuições e relações entre variáveis. Como um gráfico Seaborn *é* um gráfico Matplotlib por baixo dos panos, todas as personalizações da aula anterior continuam funcionando.

- **Plotly** segue uma filosofia diferente: seus gráficos são **interativos** — é possível dar zoom, passar o mouse sobre os pontos para ver informações, e ativar/desativar séries. Isso o torna especialmente útil para exploração de dados e apresentações em formato web ou HTML.

Como na aula anterior, os exemplos usam os dados da **Penn World Table 11.0**:

| Variável | Descrição |
|----------|-----------|
| `country` | Nome do país |
| `year` | Ano |
| `pop` | População (milhões) |
| `emp` | Pessoas empregadas (milhões) |
| `avh` | Horas médias trabalhadas por trabalhador no ano |
| `rgdpna` | PIB real (milhões de USD de 2017) |

In [ ]:
columns_to_read = ['country', 'year', 'pop', 'emp', 'rgdpna', 'avh']
df_pwt = pd.read_csv('pwt110.csv', usecols=columns_to_read)
df_pwt['pibpc'] = df_pwt['rgdpna'] / df_pwt['pop']
df_pwt.info()

## Seaborn: visualizações estatísticas <a id="seaborn"></a>

O Seaborn é importado com a abreviação convencional `sns`. Antes de criar qualquer gráfico, podemos definir o **tema** global com `sns.set_theme()`, o que já elimina a aparência padrão bastante crua do Matplotlib.

### Temas e paletas <a id="temas"></a>

O Seaborn oferece cinco temas pré-definidos que controlam o fundo e a grade do gráfico:

| Tema | Descrição |
|------|-----------|
| `'white'` | Fundo branco, sem grade |
| `'whitegrid'` | Fundo branco com grade cinza |
| `'ticks'` | Fundo branco com marcações nos eixos |
| `'dark'` | Fundo escuro |
| `'darkgrid'` | Fundo escuro com grade |

Além do tema, o argumento `palette=` controla as cores das séries (`'deep'`, `'muted'`, `'Set1'`, `'Set2'`, entre outras). Para ver a diferença entre os temas:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = list(range(2000, 2021, 5))
y = [8.5, 8.2, 7.9, 7.6, 7.4]

for ax, style in zip(axes, ['white', 'whitegrid', 'ticks']):
    sns.set_theme(style=style)
    ax.plot(x, y, marker='o', color='steelblue', linewidth=2)
    ax.set_title(f"style='{style}'", fontsize=11)
    ax.set_xlabel('Ano')
    ax.set_ylabel('Horas / dia útil')

plt.suptitle('Comparação de temas do Seaborn', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Voltamos ao tema padrão para o restante da aula
sns.set_theme(style='white')

### Gráfico de linha <a id="lineplot"></a>

Para criar um gráfico de linha com o Seaborn, usamos `sns.lineplot()`. A diferença em relação ao `ax.plot()` do Matplotlib é que passamos o **DataFrame inteiro** (`data=`) e indicamos os nomes das colunas como strings. Isso torna o código mais legível:

In [ ]:
df_pwtbr = df_pwt.loc[df_pwt['country'] == 'Brazil'].copy()
df_pwtbr['avh_dia'] = df_pwtbr['avh'] / 260

sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(9, 6))

sns.lineplot(data=df_pwtbr, x='year', y='avh_dia',
             ax=ax, color='#1f77b4', linewidth=2)

ax.set_title('Horas médias trabalhadas por dia útil — Brasil', fontsize=13)
ax.set_xlabel('Ano')
ax.set_ylabel('Horas por dia útil')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

O resultado é idêntico ao que produziríamos com o Matplotlib, mas o código é mais direto. Repare que continuamos usando os métodos do Matplotlib para ajustar o gráfico (`ax.set_title`, `ax.spines` etc.) — as duas bibliotecas trabalham juntas.

Uma das principais vantagens do Seaborn aparece quando queremos comparar **múltiplos grupos**: basta usar o argumento `hue=` com o nome da coluna que define os grupos, e o Seaborn plota cada grupo em uma cor diferente e gera a legenda automaticamente:

In [ ]:
countries = ['Brazil', 'United States', 'Germany', 'Japan', 'China']
df_multi = df_pwt[df_pwt['country'].isin(countries)].copy()
df_multi['avh_dia'] = df_multi['avh'] / 260
df_multi['country'] = df_multi['country'].replace({'United States': 'EUA'})

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))

sns.lineplot(data=df_multi.dropna(subset=['avh_dia']),
             x='year', y='avh_dia',
             hue='country',
             ax=ax, linewidth=1.8)

ax.set_title('Horas médias trabalhadas por dia útil — países selecionados', fontsize=13)
ax.set_xlabel('Ano')
ax.set_ylabel('Horas por dia útil')
ax.legend(title='País', bbox_to_anchor=(1.02, 1), loc='upper left', framealpha=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

Com apenas o argumento `hue='country'`, o Seaborn plotou cinco países em cores distintas e criou a legenda automaticamente — o que exigiria um loop e muito mais código no Matplotlib puro.

### Gráfico de dispersão <a id="scatterplot"></a>

Na aula anterior criamos um gráfico de dispersão entre horas trabalhadas e PIB per capita. Com `sns.scatterplot()` podemos ir além: o argumento `hue=` adiciona uma **terceira dimensão de informação** diferenciando os pontos por cor.

Vamos classificar os países de 2010 em três grupos de renda com base no PIB per capita:

In [ ]:
df_2010 = df_pwt[df_pwt['year'] == 2010].dropna(subset=['avh', 'pibpc']).copy()
df_2010['log_pibpc'] = np.log(df_2010['pibpc'])

# Classificar países em três grupos de renda por tercis do PIB per capita
t33, t67 = df_2010['pibpc'].quantile([1/3, 2/3])

def grupo_renda(x):
    if x <= t33:
        return 'Baixa renda'
    elif x <= t67:
        return 'Renda média'
    else:
        return 'Alta renda'

df_2010['grupo'] = df_2010['pibpc'].apply(grupo_renda)
print(df_2010['grupo'].value_counts())

In [ ]:
palette = {'Alta renda': '#2ca02c', 'Renda média': '#ff7f0e', 'Baixa renda': '#d62728'}

sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df_2010,
    x='avh', y='log_pibpc',
    hue='grupo',
    hue_order=['Alta renda', 'Renda média', 'Baixa renda'],
    palette=palette,
    alpha=0.75,
    s=60,
    ax=ax
)

ax.set_title('Horas trabalhadas vs. PIB per capita — 2010', fontsize=13)
ax.set_xlabel('Horas médias trabalhadas no ano (avh)')
ax.set_ylabel('Log do PIB per capita (USD 2017)')
ax.legend(title='Grupo de renda', framealpha=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

Os países de **alta renda** (verde) tendem a se concentrar na parte inferior do eixo X (menos horas trabalhadas) e superior do eixo Y (maior PIB per capita), evidenciando o padrão discutido na literatura sobre desenvolvimento econômico.

### Distribuições: histograma e KDE <a id="distribuicoes"></a>

O Seaborn tem ferramentas específicas para visualizar distribuições de variáveis numéricas:

- `sns.histplot()`: histograma, contando observações por intervalo
- `sns.kdeplot()`: estimação por *kernel* de densidade — uma versão contínua e suavizada do histograma

#### Histograma

In [ ]:
sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=df_2010,
    x='avh',
    bins=25,
    color='steelblue',
    edgecolor='white',
    ax=ax
)

ax.set_title('Distribuição das horas trabalhadas entre países — 2010', fontsize=13)
ax.set_xlabel('Horas médias trabalhadas no ano (avh)')
ax.set_ylabel('Número de países')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

O parâmetro `bins=` controla o número de intervalos. Um valor muito baixo cria um histograma grosseiro; muito alto, intervalos esparsos.

#### Estimador de densidade (KDE)

O `sns.kdeplot()` produz uma curva contínua que estima a distribuição dos dados. Uma das suas aplicações mais úteis é **sobrepor a distribuição de múltiplos grupos** no mesmo gráfico, facilitando a comparação:

In [ ]:
sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(9, 5))

for grupo, cor in zip(['Alta renda', 'Renda média', 'Baixa renda'],
                      ['#2ca02c',    '#ff7f0e',     '#d62728']):
    subset = df_2010[df_2010['grupo'] == grupo]
    sns.kdeplot(data=subset, x='avh', label=grupo, color=cor, linewidth=2, ax=ax)

ax.set_title('Horas trabalhadas por grupo de renda — 2010', fontsize=13)
ax.set_xlabel('Horas médias trabalhadas no ano (avh)')
ax.set_ylabel('Densidade')
ax.legend(title='Grupo de renda', framealpha=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

O KDE deixa evidente que a distribuição das horas trabalhadas difere entre grupos de renda: países ricos concentram-se em jornadas mais curtas.

### Boxplot <a id="boxplot"></a>

O **boxplot** (diagrama de caixa) é uma forma compacta de visualizar distribuições: mostra a mediana (linha central), os quartis (a caixa) e os valores atípicos (*outliers*, pontos isolados). Com `sns.boxplot()` podemos comparar distribuições entre grupos com uma única linha de código.

Vamos ver como as horas trabalhadas no Brasil variaram por década — cada caixa resume os anos daquela década:

In [ ]:
df_pwtbr_dec = df_pwt[df_pwt['country'] == 'Brazil'].copy()
df_pwtbr_dec['decada'] = (df_pwtbr_dec['year'] // 10 * 10).astype(str) + 's'
df_pwtbr_dec['avh_dia'] = df_pwtbr_dec['avh'] / 260

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    data=df_pwtbr_dec.dropna(subset=['avh_dia']),
    x='decada',
    y='avh_dia',
    hue='decada',
    palette='Blues',
    legend=False,
    ax=ax
)

ax.set_title('Horas trabalhadas por dia útil no Brasil, por década', fontsize=12)
ax.set_xlabel('Década')
ax.set_ylabel('Horas por dia útil')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

Cada caixa agrupa as observações anuais de uma década. A queda clara da mediana ao longo do tempo confirma a tendência de redução da jornada de trabalho no Brasil.

### Mapa de calor (Heatmap) <a id="heatmap"></a>

O `sns.heatmap()` é especialmente útil para visualizar **matrizes de correlação**: usa uma escala de cores para representar o coeficiente de correlação entre cada par de variáveis. O parâmetro `annot=True` exibe os valores numéricos dentro de cada célula:

In [ ]:
df_corr = df_pwt[df_pwt['year'] == 2010].dropna(subset=['pop', 'emp', 'avh', 'pibpc']).copy()
df_corr['log_pibpc'] = np.log(df_corr['pibpc'])

variaveis = df_corr[['pop', 'emp', 'avh', 'log_pibpc']].copy()
variaveis.columns = ['Pop. (mi)', 'Emp. (mi)', 'Horas (avh)', 'Log PIB/cápita']
corr = variaveis.corr()

sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(7, 6))

sns.heatmap(
    corr,
    annot=True,          # exibe os valores numéricos
    fmt='.2f',           # 2 casas decimais
    cmap='coolwarm',     # paleta divergente: azul (-1) → branco (0) → vermelho (+1)
    vmin=-1, vmax=1,
    square=True,         # células quadradas
    linewidths=0.5,
    ax=ax
)

ax.set_title('Correlação entre variáveis da PWT — 2010', fontsize=13)
plt.tight_layout()
plt.show()

A correlação negativa entre horas trabalhadas e log PIB per capita (`-0.50`) confirma o padrão que observamos nos gráficos anteriores.

---

## Plotly: gráficos interativos <a id="plotly"></a>

Até agora todos os gráficos são **estáticos** — imagens fixas. O **Plotly** produz gráficos **interativos** em HTML: é possível passar o mouse sobre os pontos para ver os valores, dar zoom em regiões específicas e clicar na legenda para ocultar ou mostrar séries.

A interface mais simples é o módulo `plotly.express` (importado como `px`), que funciona de forma análoga ao Seaborn: passamos um DataFrame e indicamos as colunas para cada eixo.

> **Nota:** No VS Code, gráficos Plotly são exibidos inline no notebook. Em outros ambientes, pode ser necessário chamar `fig.show()` explicitamente.

### Gráfico de linha interativo <a id="plotly-linha"></a>

Vamos começar reproduzindo o gráfico de horas trabalhadas no Brasil — mas desta vez interativo:

In [ ]:
fig = px.line(
    df_pwtbr.dropna(subset=['avh_dia']),
    x='year',
    y='avh_dia',
    title='Horas médias trabalhadas por dia útil — Brasil',
    labels={'year': 'Ano', 'avh_dia': 'Horas por dia útil'},
    template='simple_white'
)

fig.show()

Passe o mouse sobre a linha para ver o valor exato de cada ano. No canto superior direito há ferramentas de zoom, seleção e download da imagem.

### Múltiplos países <a id="plotly-multi"></a>

Assim como no Seaborn, o argumento `color=` separa automaticamente os grupos — e o Plotly adiciona interatividade na legenda:

In [ ]:
countries_sel = ['Brazil', 'United States', 'Germany', 'Japan', 'China']
df_plotly_multi = df_pwt[df_pwt['country'].isin(countries_sel)].copy()
df_plotly_multi['avh_dia'] = df_plotly_multi['avh'] / 260
df_plotly_multi['country'] = df_plotly_multi['country'].replace({'United States': 'EUA'})

fig = px.line(
    df_plotly_multi.dropna(subset=['avh_dia']),
    x='year',
    y='avh_dia',
    color='country',
    title='Horas médias trabalhadas por dia útil — países selecionados',
    labels={'year': 'Ano', 'avh_dia': 'Horas por dia útil', 'country': 'País'},
    template='simple_white'
)

fig.update_layout(legend_title_text='País')
fig.show()

Clique nos nomes na legenda para ativar ou desativar países. Dê um duplo clique para isolar apenas aquele país.

### Dispersão interativa <a id="plotly-scatter"></a>

O gráfico de dispersão interativo é especialmente útil porque permite **identificar países individualmente** ao passar o mouse. O argumento `hover_name=` define o rótulo principal, e `hover_data=` adiciona campos extras:

In [ ]:
fig = px.scatter(
    df_2010.dropna(subset=['avh', 'log_pibpc']),
    x='avh',
    y='log_pibpc',
    color='grupo',
    color_discrete_map={
        'Alta renda':  '#2ca02c',
        'Renda média': '#ff7f0e',
        'Baixa renda': '#d62728'
    },
    hover_name='country',
    hover_data={'avh': True, 'log_pibpc': ':.2f', 'grupo': False},
    title='Horas trabalhadas vs. PIB per capita — 2010',
    labels={
        'avh':       'Horas médias trabalhadas no ano',
        'log_pibpc': 'Log do PIB per capita',
        'grupo':     'Grupo de renda'
    },
    template='simple_white'
)

fig.show()

Passe o mouse sobre os pontos para ver o nome do país e seus valores exatos. Isso seria impossível em um gráfico estático com 100+ países.

### Gráfico de barras interativo <a id="plotly-bar"></a>

Por fim, um gráfico de barras com os 10 países de maior PIB per capita em 2010, mostrando as horas trabalhadas:

In [ ]:
df_top10_p = df_pwt[df_pwt['year'] == 2010].dropna(subset=['pibpc', 'avh']).copy()
df_top10_p = df_top10_p.sort_values('pibpc', ascending=False).iloc[:10]
df_top10_p['country'] = df_top10_p['country'].replace({
    'United Arab Emirates': 'UAE',
    'United States': 'EUA'
})
df_top10_p = df_top10_p.sort_values('avh', ascending=True)

fig = px.bar(
    df_top10_p,
    x='avh',
    y='country',
    orientation='h',
    color='avh',
    color_continuous_scale='Blues',
    hover_data={'pibpc': ':,.0f', 'avh': True},
    title='Horas trabalhadas — 10 países com maior PIB per capita (2010)',
    labels={
        'avh': 'Horas médias trabalhadas no ano',
        'country': 'País',
        'pibpc': 'PIB per capita (USD 2017)'
    },
    template='simple_white'
)

fig.update_layout(coloraxis_showscale=False)
fig.show()